In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Drive is NOT mounted!"

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
subprocess.run("pip -q install kaggle pydicom", shell=True, check=True)

RAW = Path("/content/rsna")
if not (RAW / "stage_2_train_images").exists():
    subprocess.run("mkdir -p /content/rsna", shell=True, check=True)
    subprocess.run("kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/rsna", shell=True, check=True)
    subprocess.run("unzip -q /content/rsna/rsna-pneumonia-detection-challenge.zip -d /content/rsna", shell=True, check=True)
print("Train images:", len(list((RAW / "stage_2_train_images").glob("*.dcm"))))   # expect 26684

Train images: 26684


In [ ]:
import numpy as np, pandas as pd, pydicom
from concurrent.futures import ThreadPoolExecutor

BASE = Path("/content/drive/MyDrive/ai-medical-imaging")
CACHE, MODELS = BASE / "cache", BASE / "models"
ids = pd.read_csv(CACHE / "test_ids.csv")          # same order as X_test.npy


def header(pid):
    ds = pydicom.dcmread(RAW / "stage_2_train_images" / f"{pid}.dcm", stop_before_pixels=True)
    age = str(ds.get("PatientAge", "")).strip().rstrip("Yy")
    return dict(patientId=pid, sex=str(ds.get("PatientSex", "?")),
                age=pd.to_numeric(age, errors="coerce"), view=str(ds.get("ViewPosition", "?")))


with ThreadPoolExecutor(8) as ex:
    meta = pd.DataFrame(list(ex.map(header, ids.patientId)))
df = ids.merge(meta, on="patientId")
print(len(df), "test patients")
print(df.age.describe().round(1).to_string())
print(df.sex.value_counts().to_string()); print(df.view.value_counts().to_string())

4003 test patients
count    4003.0
mean       46.7
std        16.7
min         2.0
25%        34.0
50%        49.0
75%        59.0
max        91.0
sex
M    2322
F    1681
view
PA    2170
AP    1833


In [ ]:
import torch, torch.nn as nn
import torchvision.transforms as T
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X = np.load(CACHE / "X_test.npy"); y = np.load(CACHE / "y_test.npy")
assert len(X) == len(df) and np.array_equal(y, df.label.values.astype(np.float32)), "order mismatch"

model = models.densenet121(weights=None)
model.classifier = nn.Linear(model.classifier.in_features, 1)
model.load_state_dict(torch.load(MODELS / "best_model.pt", map_location=device))
model.to(device).eval()

tf = T.Compose([T.ToPILImage(), T.Grayscale(num_output_channels=3), T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
probs = []
with torch.no_grad():
    for i in range(0, len(X), 128):
        xb = torch.stack([tf(a) for a in X[i:i + 128]]).to(device)
        probs.append(torch.sigmoid(model(xb)).squeeze(1).cpu())
df["score"] = torch.cat(probs).numpy()

from sklearn.metrics import roc_auc_score
print("Overall test AUC:", round(roc_auc_score(df.label, df.score), 4), "(should be about 0.887)")

Overall test AUC: 0.887 (should be about 0.887)


In [ ]:
THRESHOLD = 0.4837
rng = np.random.default_rng(0)


def stats(d):
    n = len(d); pos = int(d.label.sum()); neg = n - pos
    pred = d.score >= THRESHOLD
    tp = int((pred & (d.label == 1)).sum()); tn = int((~pred & (d.label == 0)).sum())
    out = dict(n=n, pneumonia_rate=round(pos / n, 3),
               sensitivity=round(tp / pos, 3) if pos else np.nan,
               specificity=round(tn / neg, 3) if neg else np.nan)
    if pos >= 10 and neg >= 10:
        boots = []
        for _ in range(300):
            s = d.sample(n, replace=True, random_state=int(rng.integers(1e9)))
            if s.label.nunique() == 2:
                boots.append(roc_auc_score(s.label, s.score))
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out["AUC"] = round(roc_auc_score(d.label, d.score), 3); out["AUC_95CI"] = f"{lo:.3f}-{hi:.3f}"
    else:
        out["AUC"] = np.nan; out["AUC_95CI"] = "too few cases"
    return out


df["age_group"] = pd.cut(df.age, [0, 39, 64, 200], labels=["<40", "40-64", "65+"])
groups = [("All", df)]
for col in ["sex", "age_group", "view"]:
    for val, d in df.groupby(col, observed=True):
        groups.append((f"{col} = {val}", d))
table = pd.DataFrame({name: stats(d) for name, d in groups}).T
print(table.to_string())
table.to_csv(BASE / "results" / "subgroup_metrics.csv")
print("saved to Drive: results/subgroup_metrics.csv")

                      n pneumonia_rate sensitivity specificity    AUC     AUC_95CI
All                4003          0.225       0.871       0.721  0.887  0.876-0.899
sex = F            1681          0.217       0.855       0.734  0.888  0.870-0.904
sex = M            2322          0.231       0.883       0.711  0.886  0.870-0.899
age_group = <40    1309          0.245       0.888        0.71    0.9  0.883-0.916
age_group = 40-64  2151          0.215       0.868       0.733  0.887  0.873-0.901
age_group = 65+     543          0.217       0.839       0.699   0.85  0.817-0.884
view = AP          1833          0.387       0.932       0.467  0.831  0.810-0.850
view = PA          2170          0.088       0.646       0.865  0.872  0.845-0.895
saved to Drive: results/subgroup_metrics.csv


In [ ]:
from sklearn.metrics import roc_curve
import json

# 1) Validation predictions (thresholds are chosen on validation; test stays untouched)
ids_v = pd.read_csv(CACHE / "val_ids.csv")
with ThreadPoolExecutor(8) as ex:
    meta_v = pd.DataFrame(list(ex.map(header, ids_v.patientId)))
dv = ids_v.merge(meta_v, on="patientId")
Xv = np.load(CACHE / "X_val.npy"); yv = np.load(CACHE / "y_val.npy")
assert len(Xv) == len(dv) and np.array_equal(yv, dv.label.values.astype(np.float32)), "order mismatch"

probs = []
with torch.no_grad():
    for i in range(0, len(Xv), 128):
        xb = torch.stack([tf(a) for a in Xv[i:i + 128]]).to(device)
        probs.append(torch.sigmoid(model(xb)).squeeze(1).cpu())
dv["score"] = torch.cat(probs).numpy()
print("Validation AUC:", round(roc_auc_score(dv.label, dv.score), 4), "(about 0.876 expected)")
print(dv["view"].value_counts().to_string())


# 2) One threshold per view: the point with the best sensitivity + specificity balance
def operating_point(d):
    fpr, tpr, thr = roc_curve(d.label, d.score)
    return float(thr[np.argmax(tpr - fpr)])


def metrics_at(d, t):
    pred = d.score >= t
    tp = int((pred & (d.label == 1)).sum()); fn = int((~pred & (d.label == 1)).sum())
    tn = int((~pred & (d.label == 0)).sum()); fp = int((pred & (d.label == 0)).sum())
    return dict(n=len(d), sensitivity=round(tp / (tp + fn), 3), specificity=round(tn / (tn + fp), 3),
                precision=round(tp / (tp + fp), 3) if tp + fp else np.nan,
                share_flagged=round((tp + fp) / len(d), 3))


view_thr = {v: operating_point(dv[dv["view"] == v]) for v in ["AP", "PA"]}
print("Thresholds chosen on validation:", {k: round(t, 4) for k, t in view_thr.items()})

# 3) Compare on the TEST set
rows = {}
for v in ["AP", "PA"]:
    d = df[df["view"] == v]
    rows[f"{v}: one threshold ({THRESHOLD})"] = metrics_at(d, THRESHOLD)
    rows[f"{v}: own threshold ({view_thr[v]:.3f})"] = metrics_at(d, view_thr[v])
rows[f"ALL: one threshold ({THRESHOLD})"] = metrics_at(df, THRESHOLD)
rows["ALL: view-specific thresholds"] = metrics_at(df, df["view"].map(view_thr))
table2 = pd.DataFrame(rows).T
print(table2.to_string())
table2.to_csv(BASE / "results" / "view_threshold_experiment.csv")
json.dump(view_thr, open(BASE / "results" / "view_thresholds.json", "w"))
print("saved to Drive: results/view_threshold_experiment.csv and view_thresholds.json")

Validation AUC: 0.8764 (about 0.876 expected)
view
PA    2196
AP    1807
Thresholds chosen on validation: {'AP': 0.7517, 'PA': 0.2488}
                                    n  sensitivity  specificity  precision  share_flagged
AP: one threshold (0.4837)     1833.0        0.932        0.467      0.525          0.687
AP: own threshold (0.752)      1833.0        0.801        0.683      0.615          0.505
PA: one threshold (0.4837)     2170.0        0.646        0.865      0.317          0.180
PA: own threshold (0.249)      2170.0        0.859        0.748      0.249          0.306
ALL: one threshold (0.4837)    4003.0        0.871        0.721      0.476          0.412
ALL: view-specific thresholds  4003.0        0.814        0.725      0.462          0.397
saved to Drive: results/view_threshold_experiment.csv and view_thresholds.json
